# CoT Generation Pipeline

**Phase 1** — Answer-only benchmark across frontier models (Claude Opus, Gemini 3.1 Pro)  
**Phase 2** — Tiered CoT generation: genuine CoT from correct answers + correction-guided CoT from wrong answers  
**Export** — Final `cot_traces.csv` ready for Kaggle training dataset upload

Each phase checkpoints per-model results so you can resume after quota/timeout.

In [ ]:
import os
import re
import unicodedata
from glob import glob
from pathlib import Path

import pandas as pd
import shutil
import kaggle_benchmarks as kbench

os.environ["RENDER_SUBRUNS"] = "False"

# =========================
# CONFIG
# =========================

SAMPLE_FILE_NAME = "benchmark_sample_500.csv"
ID_COLUMN = "id"
QUESTION_COLUMN = "prompt"
ANSWER_COLUMN = "answer"

N_JOBS = 4
TIMEOUT_SECONDS = None
MAX_ATTEMPTS = 50
RETRY_DELAY_SECONDS = 15

# Phase 1+2 use only the two models that actually work.
# Family assignment: each family goes to the model that scores highest on it.
MODEL_CANDIDATES = {
    "claude_opus_4_6": ["anthropic/claude-opus-4-6@default"],
    "gemini_3_1_pro": ["google/gemini-3.1-pro-preview"],
}

# Which model handles which families for CoT generation (Phase 2).
# Based on Phase 1 benchmark results:
#   Claude best at: encryption (60.7%), unit_conversion (54.2%), gravity (16.9%), numeral (100%)
#   Gemini best at: bit_manipulation (34.5%), equations (19.3%)
FAMILY_MODEL_MAP = {
    "encryption":       "claude_opus_4_6",
    "unit_conversion":  "claude_opus_4_6",
    "gravity":          "claude_opus_4_6",
    "numeral":          "claude_opus_4_6",
    "bit_manipulation": "gemini_3_1_pro",
    "equations":        "gemini_3_1_pro",
}

OUTPUT_DIR = (
    Path("/kaggle/working/benchmark_outputs")
    if Path("/kaggle/working").exists()
    else Path("benchmark_outputs")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COT_DIR = OUTPUT_DIR / "cot_traces"
COT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# LOAD SAMPLE + HELPERS
# =========================

def find_sample_file(file_name: str) -> str:
    candidates = sorted(glob(f"/kaggle/input/**/{file_name}", recursive=True))
    if candidates:
        return candidates[0]
    local = Path("datasets") / file_name
    if local.exists():
        return str(local.resolve())
    raise FileNotFoundError(
        f"Could not find {file_name}. On Kaggle: add a dataset containing it. "
        f"Locally: place at datasets/{file_name}."
    )

sample_path = find_sample_file(SAMPLE_FILE_NAME)
raw_df = pd.read_csv(sample_path)

required = {QUESTION_COLUMN, ANSWER_COLUMN}
missing = required - set(raw_df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

if ID_COLUMN not in raw_df.columns:
    raw_df[ID_COLUMN] = range(len(raw_df))

BENCHMARK_DF = raw_df[[ID_COLUMN, QUESTION_COLUMN, ANSWER_COLUMN]].copy()
BENCHMARK_DF = BENCHMARK_DF.rename(columns={
    ID_COLUMN: "row_id", QUESTION_COLUMN: "question", ANSWER_COLUMN: "answer",
})
for c in ["row_id", "question", "answer"]:
    BENCHMARK_DF[c] = BENCHMARK_DF[c].astype(str)

print(f"Loaded: {sample_path}  ({len(BENCHMARK_DF)} rows)")
display(BENCHMARK_DF.head(3))


def normalize_text(x) -> str:
    if pd.isna(x):
        return ""
    text = unicodedata.normalize("NFKC", str(x)).strip()
    return re.sub(r"\s+", " ", text).casefold()

def exact_match(pred: str, gold: str) -> bool:
    return str(pred).strip() == str(gold).strip()

def normalized_match(pred: str, gold: str) -> bool:
    return normalize_text(pred) == normalize_text(gold)

def classify_family(prompt_text: str) -> str:
    p = str(prompt_text).lower()
    if "bit manipulation" in p:        return "bit_manipulation"
    if "gravitational constant" in p:   return "gravity"
    if "encryption rules" in p or "encryption" in p: return "encryption"
    if "numeral system" in p:           return "numeral"
    if "unit conversion" in p or "converted" in p:   return "unit_conversion"
    if "transformation rules" in p or "equation" in p or "operator" in p: return "equations"
    return "unknown"

def resolve_model_key(candidate_keys):
    for key in candidate_keys:
        try:
            _ = kbench.llms[key]
            return key
        except Exception:
            pass
    return None

def show_available_models():
    try:
        for k in sorted(kbench.llms.keys()):
            print(" -", k)
    except Exception as e:
        print("Could not list models:", e)

def flatten_runs(runs) -> pd.DataFrame:
    runs_df = runs.as_dataframe().copy()
    def safe_result_dict(x):
        if isinstance(x, dict):
            return x
        return {"row_id": None, "question": None, "gold_answer": None,
                "prediction": "", "exact_match": False, "normalized_match": False,
                "task_error": str(x)}
    result_df = pd.json_normalize(runs_df["result"].apply(safe_result_dict))
    meta = runs_df.drop(columns=["result"]).reset_index(drop=True)
    _overlap = [c for c in meta.columns if c in result_df.columns]
    if _overlap:
        meta = meta.drop(columns=_overlap)
    flat_df = pd.concat([meta, result_df.reset_index(drop=True)], axis=1)
    return flat_df.loc[:, ~flat_df.columns.duplicated(keep="last")]

In [ ]:
# =====================================================================
# PHASE 1: Answer-only benchmark (per-model accuracy)
# =====================================================================

@kbench.task(store_task=False, name="fixed_sample_row_check")
def fixed_sample_row_check(llm, row_id: str, question: str, answer: str) -> dict:
    prompt = (
        f"{question}\n\n"
        "Return only the final answer. No explanation, no steps, no extra words."
    )
    try:
        response = llm.prompt(prompt)
        prediction = "" if response is None else str(response).strip()
        error = ""
    except Exception as e:
        prediction = ""
        error = repr(e)

    gold = "" if answer is None else str(answer).strip()
    return {
        "row_id": row_id,
        "question": question,
        "gold_answer": gold,
        "prediction": prediction,
        "exact_match": exact_match(prediction, gold) if not error else False,
        "normalized_match": normalized_match(prediction, gold) if not error else False,
        "task_error": error,
    }


# --- Resolve models ---
resolved_models = {}
for label, candidates in MODEL_CANDIDATES.items():
    resolved = resolve_model_key(candidates)
    if resolved:
        resolved_models[label] = resolved
    else:
        print(f"WARN: could not resolve {label}: {candidates}")

print("Resolved models:")
for label, key in resolved_models.items():
    print(f"  {label}: {key}")

# --- Run Phase 1 with per-model caching ---
SAMPLE_STEM = Path(SAMPLE_FILE_NAME).stem
PER_MODEL_CACHE_DIR = OUTPUT_DIR / "per_model_row_cache"
PER_MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if Path("/kaggle/input").exists():
    for _src in Path("/kaggle/input").rglob(f"*__{SAMPLE_STEM}.csv"):
        _dst = PER_MODEL_CACHE_DIR / _src.name
        if not _dst.exists():
            shutil.copy2(_src, _dst)
            print("Restored cache:", _dst)

all_model_outputs = []
for model_label, model_key in resolved_models.items():
    _cache_csv = PER_MODEL_CACHE_DIR / f"{model_label}__{SAMPLE_STEM}.csv"
    if _cache_csv.exists():
        _cached = pd.read_csv(_cache_csv)
        if len(_cached) == len(BENCHMARK_DF):
            print(f"\nSkipping {model_label} (cached, {len(_cached)} rows)")
            all_model_outputs.append(_cached)
            continue

    print(f"\nRunning Phase 1 for {model_label} -> {model_key}")
    with kbench.client.enable_cache():
        runs = fixed_sample_row_check.evaluate(
            llm=[kbench.llms[model_key]],
            evaluation_data=BENCHMARK_DF,
            stop_condition=lambda runs: len(runs) == len(BENCHMARK_DF),
            max_attempts=MAX_ATTEMPTS,
            retry_delay=RETRY_DELAY_SECONDS,
            n_jobs=N_JOBS,
            timeout=TIMEOUT_SECONDS,
            remove_run_files=True,
        )
    model_df = flatten_runs(runs)
    model_df["model_label"] = model_label
    model_df["model_key"] = model_key
    model_df.to_csv(_cache_csv, index=False)
    print(f"Saved cache: {_cache_csv}")
    all_model_outputs.append(model_df)

if not all_model_outputs:
    raise RuntimeError("No models resolved. Run show_available_models().")

all_results_df = pd.concat(all_model_outputs, ignore_index=True)
all_results_df = all_results_df.loc[:, ~all_results_df.columns.duplicated(keep="last")]
all_results_df["family"] = all_results_df["question"].map(classify_family)

all_results_df.to_csv(OUTPUT_DIR / "all_row_level_results.csv", index=False)
print(f"\nPhase 1 complete. {len(all_results_df)} total rows across {len(resolved_models)} models.")

In [ ]:
# =====================================================================
# PHASE 1 RESULTS — Leaderboard + per-family breakdown
# =====================================================================

leaderboard_df = (
    all_results_df.groupby(["model_label", "model_key"], as_index=False)
    .agg(
        rows_scored=("row_id", "count"),
        exact_matches=("exact_match", "sum"),
        normalized_matches=("normalized_match", "sum"),
        errors=("task_error", lambda s: int((s != "").sum())),
    )
)
leaderboard_df["exact_match_pct"] = (100.0 * leaderboard_df["exact_matches"] / leaderboard_df["rows_scored"]).round(2)
leaderboard_df["normalized_match_pct"] = (100.0 * leaderboard_df["normalized_matches"] / leaderboard_df["rows_scored"]).round(2)
leaderboard_df = leaderboard_df.sort_values("normalized_match_pct", ascending=False).reset_index(drop=True)
leaderboard_df.to_csv(OUTPUT_DIR / "leaderboard.csv", index=False)

print("Overall leaderboard:")
display(leaderboard_df)

family_breakdown = (
    all_results_df.groupby(["model_label", "family"], as_index=False)
    .agg(
        rows=("row_id", "count"),
        norm_matches=("normalized_match", "sum"),
    )
)
family_breakdown["norm_match_pct"] = (100.0 * family_breakdown["norm_matches"] / family_breakdown["rows"]).round(1)

pivot = family_breakdown.pivot_table(index="family", columns="model_label", values="norm_match_pct", aggfunc="first").sort_index()
print("\nNormalized match % by family:")
display(pivot)

In [ ]:
# =====================================================================
# PHASE 2: Tiered CoT Generation
#
# Tier 1 (Genuine CoT): rows the assigned model got RIGHT in Phase 1.
#   -> Ask model to solve with full step-by-step reasoning.
#   -> Keep only if extracted \boxed{} matches gold answer.
#
# Tier 2 (Correction-guided CoT): rows the model got WRONG or errored.
#   -> Give model the correct answer, ask it to generate reasoning.
#   -> Keep only if extracted \boxed{} matches gold answer.
# =====================================================================

COT_PROMPT_TIER1 = (
    "{question}\n\n"
    "Think through this step by step. Show your complete reasoning process, "
    "then put your final answer inside \\boxed{{}}. "
    "For example: \\boxed{{your answer}}"
)

COT_PROMPT_TIER2 = (
    "{question}\n\n"
    "Your previous answer was incorrect. The objectively correct answer is: {gold_answer}\n\n"
    "Recalibrate your thinking. Generate a rigorous, step-by-step Chain of Thought "
    "that logically and flawlessly arrives at this correct answer. "
    "Explicitly point out the hidden trap or complex step where a system might typically fail. "
    "Put your final answer inside \\boxed{{}}. For example: \\boxed{{your answer}}"
)

def extract_boxed(text: str) -> str:
    if not text:
        return ""
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else ""
    return ""


@kbench.task(store_task=False, name="cot_tier1_generate")
def cot_tier1_generate(llm, row_id: str, question: str, answer: str) -> dict:
    """Tier 1: genuine CoT — model solves from scratch with reasoning."""
    prompt = COT_PROMPT_TIER1.format(question=question)
    try:
        response = llm.prompt(prompt)
        cot_text = "" if response is None else str(response).strip()
        error = ""
    except Exception as e:
        cot_text = ""
        error = repr(e)

    gold = str(answer).strip()
    extracted = extract_boxed(cot_text)
    verified = normalized_match(extracted, gold) if not error else False

    return {
        "row_id": row_id,
        "question": question,
        "gold_answer": gold,
        "cot_response": cot_text,
        "extracted_answer": extracted,
        "verified": verified,
        "tier": "tier1_genuine",
        "task_error": error,
    }


@kbench.task(store_task=False, name="cot_tier2_correction")
def cot_tier2_correction(llm, row_id: str, question: str, answer: str) -> dict:
    """Tier 2: correction-guided CoT — model given correct answer, asked to reason toward it."""
    prompt = COT_PROMPT_TIER2.format(question=question, gold_answer=str(answer).strip())
    try:
        response = llm.prompt(prompt)
        cot_text = "" if response is None else str(response).strip()
        error = ""
    except Exception as e:
        cot_text = ""
        error = repr(e)

    gold = str(answer).strip()
    extracted = extract_boxed(cot_text)
    verified = normalized_match(extracted, gold) if not error else False

    return {
        "row_id": row_id,
        "question": question,
        "gold_answer": gold,
        "cot_response": cot_text,
        "extracted_answer": extracted,
        "verified": verified,
        "tier": "tier2_correction",
        "task_error": error,
    }

print("Phase 2 tasks defined. Ready to run CoT generation.")

In [ ]:
# =====================================================================
# PHASE 2 EXECUTION — Run CoT generation per family with assigned model
# =====================================================================

BENCHMARK_DF["family"] = BENCHMARK_DF["question"].map(classify_family)

# Build per-row Phase 1 outcome lookup: did the assigned model get this row right?
phase1_correct_set = set()
for model_label in resolved_models:
    model_rows = all_results_df[all_results_df["model_label"] == model_label]
    correct_ids = set(model_rows[model_rows["normalized_match"] == True]["row_id"].astype(str))
    phase1_correct_set.update((model_label, rid) for rid in correct_ids)

tier1_rows = []  # rows where assigned model got it right -> genuine CoT
tier2_rows = []  # rows where assigned model got it wrong -> correction CoT

for _, row in BENCHMARK_DF.iterrows():
    fam = row["family"]
    assigned_model = FAMILY_MODEL_MAP.get(fam)
    if assigned_model is None or assigned_model not in resolved_models:
        continue
    entry = {"row_id": row["row_id"], "question": row["question"], "answer": row["answer"]}
    if (assigned_model, row["row_id"]) in phase1_correct_set:
        tier1_rows.append((assigned_model, entry))
    else:
        tier2_rows.append((assigned_model, entry))

print(f"Tier 1 (genuine CoT):      {len(tier1_rows)} rows")
print(f"Tier 2 (correction-guided): {len(tier2_rows)} rows")
print(f"Total:                      {len(tier1_rows) + len(tier2_rows)} rows")

# Group by model for batch API calls
from collections import defaultdict
tier1_by_model = defaultdict(list)
tier2_by_model = defaultdict(list)
for model_label, entry in tier1_rows:
    tier1_by_model[model_label].append(entry)
for model_label, entry in tier2_rows:
    tier2_by_model[model_label].append(entry)

for ml in resolved_models:
    print(f"  {ml}: tier1={len(tier1_by_model[ml])}, tier2={len(tier2_by_model[ml])}")

In [ ]:
# =====================================================================
# PHASE 2A: Tier 1 — Genuine CoT (model re-solves with reasoning)
# =====================================================================

all_cot_results = []

for model_label, rows in tier1_by_model.items():
    if not rows:
        continue
    cache_path = COT_DIR / f"tier1__{model_label}__{SAMPLE_STEM}.csv"

    # Restore from Kaggle input if available
    if not cache_path.exists() and Path("/kaggle/input").exists():
        for _src in Path("/kaggle/input").rglob(cache_path.name):
            shutil.copy2(_src, cache_path)
            print(f"Restored tier1 cache: {cache_path}")
            break

    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if len(cached) == len(rows):
            print(f"\nSkipping Tier 1 for {model_label} (cached, {len(cached)} rows)")
            all_cot_results.append(cached)
            continue

    print(f"\nRunning Tier 1 CoT for {model_label} ({len(rows)} rows)")
    eval_df = pd.DataFrame(rows)

    with kbench.client.enable_cache():
        runs = cot_tier1_generate.evaluate(
            llm=[kbench.llms[resolved_models[model_label]]],
            evaluation_data=eval_df,
            stop_condition=lambda runs: len(runs) == len(eval_df),
            max_attempts=MAX_ATTEMPTS,
            retry_delay=RETRY_DELAY_SECONDS,
            n_jobs=N_JOBS,
            timeout=TIMEOUT_SECONDS,
            remove_run_files=True,
        )

    result_df = flatten_runs(runs)
    result_df["model_label"] = model_label
    result_df.to_csv(cache_path, index=False)
    print(f"Saved tier1 cache: {cache_path}")
    all_cot_results.append(result_df)

if all_cot_results:
    tier1_df = pd.concat(all_cot_results, ignore_index=True)
    tier1_df = tier1_df.loc[:, ~tier1_df.columns.duplicated(keep="last")]
    verified_t1 = tier1_df["verified"].sum() if "verified" in tier1_df.columns else 0
    print(f"\nTier 1 complete: {len(tier1_df)} rows, {verified_t1} verified")
else:
    tier1_df = pd.DataFrame()
    print("No Tier 1 rows to process.")

In [ ]:
# =====================================================================
# PHASE 2B: Tier 2 — Correction-guided CoT
# =====================================================================

all_cot_t2_results = []

for model_label, rows in tier2_by_model.items():
    if not rows:
        continue
    cache_path = COT_DIR / f"tier2__{model_label}__{SAMPLE_STEM}.csv"

    if not cache_path.exists() and Path("/kaggle/input").exists():
        for _src in Path("/kaggle/input").rglob(cache_path.name):
            shutil.copy2(_src, cache_path)
            print(f"Restored tier2 cache: {cache_path}")
            break

    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if len(cached) == len(rows):
            print(f"\nSkipping Tier 2 for {model_label} (cached, {len(cached)} rows)")
            all_cot_t2_results.append(cached)
            continue

    print(f"\nRunning Tier 2 (correction) CoT for {model_label} ({len(rows)} rows)")
    eval_df = pd.DataFrame(rows)

    with kbench.client.enable_cache():
        runs = cot_tier2_correction.evaluate(
            llm=[kbench.llms[resolved_models[model_label]]],
            evaluation_data=eval_df,
            stop_condition=lambda runs: len(runs) == len(eval_df),
            max_attempts=MAX_ATTEMPTS,
            retry_delay=RETRY_DELAY_SECONDS,
            n_jobs=N_JOBS,
            timeout=TIMEOUT_SECONDS,
            remove_run_files=True,
        )

    result_df = flatten_runs(runs)
    result_df["model_label"] = model_label
    result_df.to_csv(cache_path, index=False)
    print(f"Saved tier2 cache: {cache_path}")
    all_cot_t2_results.append(result_df)

if all_cot_t2_results:
    tier2_df = pd.concat(all_cot_t2_results, ignore_index=True)
    tier2_df = tier2_df.loc[:, ~tier2_df.columns.duplicated(keep="last")]
    verified_t2 = tier2_df["verified"].sum() if "verified" in tier2_df.columns else 0
    print(f"\nTier 2 complete: {len(tier2_df)} rows, {verified_t2} verified")
else:
    tier2_df = pd.DataFrame()
    print("No Tier 2 rows to process.")

In [ ]:
# =====================================================================
# PHASE 2 SUMMARY + EXPORT
# =====================================================================

cot_parts = []
if len(tier1_df) > 0:
    cot_parts.append(tier1_df)
if len(tier2_df) > 0:
    cot_parts.append(tier2_df)

if not cot_parts:
    raise RuntimeError("No CoT results from either tier.")

combined_cot = pd.concat(cot_parts, ignore_index=True)
combined_cot = combined_cot.loc[:, ~combined_cot.columns.duplicated(keep="last")]
combined_cot["family"] = combined_cot["question"].map(classify_family)

# Filter to verified-only (extracted answer matches gold)
verified_cot = combined_cot[combined_cot["verified"] == True].copy()

print("=" * 60)
print("COT GENERATION SUMMARY")
print("=" * 60)
print(f"Total rows processed:  {len(combined_cot)}")
print(f"Verified (usable):     {len(verified_cot)}")
print(f"Discarded (bad boxed): {len(combined_cot) - len(verified_cot)}")
print()

for tier_name in ["tier1_genuine", "tier2_correction"]:
    subset = combined_cot[combined_cot["tier"] == tier_name]
    v = subset["verified"].sum() if len(subset) > 0 else 0
    print(f"  {tier_name}: {len(subset)} total, {v} verified ({100*v/max(1,len(subset)):.1f}%)")

print()
print("Per-family verified counts:")
if len(verified_cot) > 0:
    fam_counts = verified_cot.groupby(["family", "tier"]).size().unstack(fill_value=0)
    display(fam_counts)

# --- Export final training-ready CSV ---
# Columns: row_id, question (raw), gold_answer, cot_response (full reasoning), tier, family, model_label
export_cols = ["row_id", "question", "gold_answer", "cot_response", "tier", "family", "model_label"]
available_cols = [c for c in export_cols if c in verified_cot.columns]
export_df = verified_cot[available_cols].copy()

export_path = OUTPUT_DIR / "cot_traces_verified.csv"
export_df.to_csv(export_path, index=False)
print(f"\nExported {len(export_df)} verified CoT traces to: {export_path}")

# Also save full (unfiltered) for debugging
combined_cot.to_csv(OUTPUT_DIR / "cot_traces_all.csv", index=False)
print(f"Full (unfiltered) saved to: {OUTPUT_DIR / 'cot_traces_all.csv'}")
print("=" * 60)
print(f"\nDownload {export_path.name} and upload as a Kaggle dataset for training.")

## After this notebook

1. **Download** `benchmark_outputs/cot_traces_verified.csv` and the `cot_traces/` checkpoint folder  
2. **Upload** `cot_traces_verified.csv` as a Kaggle dataset  
3. **Run** the training notebook (`improved-RunA-Notebook.ipynb`) which mixes these CoT traces with the 8500 bare-answer rows  

**Resume support:** All Phase 1 + Phase 2 results are cached per-model under `benchmark_outputs/`. Upload the entire `benchmark_outputs/` folder as a Kaggle dataset input for the next run to skip completed API calls.